# 09 What the final model relies on — SHAP and permutation importance (E-03)

Written by Yiling (2026-11-15). Uses the BD-07 models: **test** (refit 2021–23, scored on 2024) once the one-time test has run, otherwise the **dry run** (fit 2021–22, scored on 2023), so the code is checked before the test. Week 6: permutation importance on the test set.

Importance describes which inputs the model uses to **predict historical decisions**. It is not a causal effect, and correlated inputs share credit, so grouped permutation is reported alongside single features.

In [1]:
import sys; sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import config
assert config.ROOT == Path.cwd().resolve().parent, "Run from the notebooks directory."
(config.REPORTS / "figures").mkdir(exist_ok=True)
from src import explain
LABEL = explain.available_label()
print("Using:", LABEL)
rows = explain.scored_rows(LABEL)
y = rows[config.TARGET_Q1].to_numpy(int)
SPECS = [("xgb", "CASE_COURT_JUDGE"), ("xgb", "CASE")]   # final full model; case-only model for comparison
print(len(rows), "scored cases")

Using: test


135821 scored cases


## SHAP (mean |SHAP| by original feature, 5,000-case sample)

In [2]:
shap_tables, per_case, samples = [], {}, {}
for model, fs in SPECS:
    pipe = explain.load_model(LABEL, model, fs)
    X = explain.raw_X(rows, fs, LABEL)
    table, values, sample = explain.shap_by_feature(pipe, X)
    table.insert(0, "feature_set", fs); table.insert(0, "model", model)
    shap_tables.append(table); per_case[fs], samples[fs] = values, sample
shap_table = pd.concat(shap_tables, ignore_index=True)
shap_table.to_csv(config.REPORTS / f"e03_{LABEL}_shap.csv", index=False)
shap_table.loc[shap_table.feature_set.eq("CASE_COURT_JUDGE")].round(4)

/opt/anaconda3/envs/pol408/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,feature_set,feature,mean_abs_shap,rank
0,xgb,CASE_COURT_JUDGE,arraign_charge_category,3.2919,1
1,xgb,CASE_COURT_JUDGE,judge_id,1.3977,2
2,xgb,CASE_COURT_JUDGE,top_charge_weight_at_arraign,0.7429,3
3,xgb,CASE_COURT_JUDGE,top_severity_at_arraign,0.6832,4
4,xgb,CASE_COURT_JUDGE,county_name,0.6822,5
5,xgb,CASE_COURT_JUDGE,supervision,0.5371,6
6,xgb,CASE_COURT_JUDGE,prior_nonvfo_cnt,0.4312,7
7,xgb,CASE_COURT_JUDGE,pend_vfo,0.4174,8
8,xgb,CASE_COURT_JUDGE,age_at_arrest,0.4056,9
9,xgb,CASE_COURT_JUDGE,prior_vfo_cnt,0.3556,10


In [3]:
t = shap_table.loc[shap_table.feature_set.eq("CASE_COURT_JUDGE")].head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.barh(t["feature"], t["mean_abs_shap"], color="#4C72B0")
ax.set_xlabel("Mean |SHAP| (log-odds), summed over one-hot columns")
ax.set_title(f"XGBoost, CASE_COURT_JUDGE — {LABEL}")
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"e03_{LABEL}_shap_bar.png", dpi=200); plt.close(fig)

Top three features: how the prediction moves with each value (mean SHAP by category; numeric features binned).

In [4]:
fs = "CASE_COURT_JUDGE"
top3 = shap_table.loc[shap_table.feature_set.eq(fs), "feature"].head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
dep_rows = []
for ax, feat in zip(axes, top3):
    x = samples[fs][feat]
    s = per_case[fs][feat]
    if pd.api.types.is_numeric_dtype(x) and x.nunique() > 12:
        key = pd.qcut(x, 10, duplicates="drop").astype(str)
    else:
        key = x.astype(str)
    d = pd.DataFrame({"value": key, "shap": s}).groupby("value")["shap"].agg(["mean", "size"])
    d = d.loc[d["size"] >= config.MIN_GROUP_N].sort_values("mean")
    ax.barh(d.index.astype(str).str.slice(0, 28), d["mean"], color="#55A868")
    ax.axvline(0, color="k", lw=0.6); ax.set_title(feat); ax.set_xlabel("Mean SHAP")
    dep_rows.append(d.assign(feature=feat).reset_index())
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"e03_{LABEL}_dependence_top3.png", dpi=200); plt.close(fig)
pd.concat(dep_rows).to_csv(config.REPORTS / f"e03_{LABEL}_dependence_top3.csv", index=False)
print("Top three:", top3)

Top three: ['arraign_charge_category', 'judge_id', 'top_charge_weight_at_arraign']


## Permutation importance (drop in AUC, 5 shuffles) and gain importance

In [5]:
perm_tables, group_tables, gain_tables = [], [], []
for model, fs in SPECS:
    pipe = explain.load_model(LABEL, model, fs)
    X = explain.raw_X(rows, fs, LABEL)
    single = explain.permutation_importance(pipe, X, y)
    grouped = explain.permutation_importance(pipe, X, y, groups=explain.FEATURE_GROUPS)
    gain = explain.gain_by_feature(pipe, X.columns)
    for t in (single, grouped, gain):
        t.insert(0, "feature_set", fs); t.insert(0, "model", model)
    perm_tables.append(single); group_tables.append(grouped); gain_tables.append(gain)
perm = pd.concat(perm_tables, ignore_index=True); groups = pd.concat(group_tables, ignore_index=True)
gain = pd.concat(gain_tables, ignore_index=True)
perm.to_csv(config.REPORTS / f"e03_{LABEL}_permutation.csv", index=False)
groups.to_csv(config.REPORTS / f"e03_{LABEL}_permutation_groups.csv", index=False)
gain.to_csv(config.REPORTS / f"e03_{LABEL}_gain.csv", index=False)
groups.round(4)

,model,feature_set,item,columns,auc_drop_mean,auc_drop_sd,base_auc,n_repeats
0,xgb,CASE_COURT_JUDGE,charge,"top_severity_at_arraign, top_charge_weight_at_...",0.2784,0.0008,0.9369,5
1,xgb,CASE_COURT_JUDGE,record_counts (D14),"prior_vfo_cnt, prior_nonvfo_cnt, prior_misd_cn...",0.0914,0.0008,0.9369,5
2,xgb,CASE_COURT_JUDGE,supervision_and_process,"supervision, dcjs_missing, dat_wo_ws_prior_to_...",0.0134,0.0004,0.9369,5
3,xgb,CASE_COURT_JUDGE,judge,judge_id,0.0082,0.0002,0.9369,5
4,xgb,CASE_COURT_JUDGE,county,county_name,0.0044,0.0001,0.9369,5
5,xgb,CASE_COURT_JUDGE,age,age_at_arrest,0.0028,0.0001,0.9369,5
6,xgb,CASE,charge,"top_severity_at_arraign, top_charge_weight_at_...",0.2837,0.0008,0.9298,5
7,xgb,CASE,record_counts (D14),"prior_vfo_cnt, prior_nonvfo_cnt, prior_misd_cn...",0.0913,0.0008,0.9298,5
8,xgb,CASE,supervision_and_process,"supervision, dcjs_missing, dat_wo_ws_prior_to_...",0.0142,0.0004,0.9298,5
9,xgb,CASE,age,age_at_arrest,0.0031,0.0002,0.9298,5


In [6]:
full = "CASE_COURT_JUDGE"
compare = (shap_table.loc[shap_table.feature_set.eq(full), ["feature", "rank"]].rename(columns={"rank": "shap_rank"})
           .merge(perm.loc[perm.feature_set.eq(full), ["item", "auc_drop_mean"]].rename(columns={"item": "feature"}), on="feature")
           .merge(gain.loc[gain.feature_set.eq(full), ["feature", "gain_share"]], on="feature"))
compare["permutation_rank"] = compare["auc_drop_mean"].rank(ascending=False, method="min").astype(int)
compare["gain_rank"] = compare["gain_share"].rank(ascending=False, method="min").astype(int)
compare = compare.sort_values("shap_rank")
compare.to_csv(config.REPORTS / f"e03_{LABEL}_rank_comparison.csv", index=False)
compare.round(4)

,feature,shap_rank,auc_drop_mean,gain_share,permutation_rank,gain_rank
0,arraign_charge_category,1,0.0332,0.0852,2,3
1,judge_id,2,0.0083,0.0576,9,5
2,top_charge_weight_at_arraign,3,0.0251,0.0753,4,4
3,top_severity_at_arraign,4,0.1364,0.3199,1,1
4,county_name,5,0.0043,0.0149,11,14
5,supervision,6,0.0013,0.0096,15,15
6,prior_nonvfo_cnt,7,0.0020,0.0152,14,13
7,pend_vfo,8,0.0093,0.0282,8,10
8,age_at_arrest,9,0.0028,0.0336,12,9
9,prior_vfo_cnt,10,0.0025,0.0172,13,11


**How to read this (for the deck):** SHAP says how much each input moves individual predictions. Permutation says how much ranking quality (AUC) is lost without it. Gain counts how much each input is used in tree splits, so it favours features with many categories. They need not agree. For policymakers, permutation on the test year is the most direct answer to "what does the model need?".